# Lezione 2 — Pensare in vettori
## *"Se stai scrivendo un for, probabilmente c'è un modo migliore"*

**Micro-Corso Python per Dottorandi/Borsisti EE**

---

### Obiettivi della lezione
- Usare NumPy per manipolare segnali senza cicli `for`
- Capire come indicizzazione, slicing e broadcasting sostituiscono il codice C
- Calcolare e interpretare la FFT di un segnale reale
- Riconoscere l'effetto della finestratura e dello zero-padding sullo spettro

### Setup
```python
pip install numpy matplotlib
```

---

## Blocco 2.1 — NumPy: l'array che mancava al C (50 min)

### Perché NumPy?

In C, per fare operazioni su un array devi scrivere un ciclo:
```c
float out[N];
for (int i = 0; i < N; i++) out[i] = sin(2*M_PI*f*t[i]);
```

In NumPy la stessa cosa è:
```python
out = np.sin(2 * np.pi * f * t)
```

Non è solo più corto: NumPy esegue il ciclo **in C compilato** → stesso risultato, ~100× più veloce di un `for` Python puro.

In [ ]:
import numpy as np

# Benchmark: somma di 1 milione di elementi
import time

N = 1_000_000

# Modo Python puro (lista)
lista = list(range(N))
t0 = time.perf_counter()
somma = sum(lista)
t1 = time.perf_counter()
print(f"Lista Python: {(t1-t0)*1000:.1f} ms")

# Modo NumPy (array)
arr = np.arange(N)
t0 = time.perf_counter()
somma = np.sum(arr)
t1 = time.perf_counter()
print(f"NumPy array: {(t1-t0)*1000:.1f} ms")

# Il rapporto di velocità dipende dall'hardware, ma NumPy è sempre molto più veloce

### Creare array: le funzioni fondamentali

In [ ]:
import numpy as np

# np.array — da lista Python
v = np.array([1.0, 2.0, 3.0, 4.0])
print(f"array:  {v}")
print(f"dtype:  {v.dtype}")    # float64 di default
print(f"shape:  {v.shape}")    # (4,) — tupla con le dimensioni

# np.zeros, np.ones — array inizializzati
z = np.zeros(5)
o = np.ones((2, 3))            # matrice 2x3
print(f"zeros:  {z}")
print(f"ones:\n{o}")

# np.linspace — N punti equispaziati tra a e b (inclusi)
t = np.linspace(0, 1, 5)
print(f"linspace(0,1,5): {t}")

# np.arange — come range() ma produce un array
# arange(start, stop, step) — stop ESCLUSO
a = np.arange(0, 1.0, 0.25)
print(f"arange:  {a}")

# Caso d'uso tipico: asse temporale per un segnale
fs = 1000          # frequenza di campionamento [Hz]
T  = 0.5           # durata [s]
t  = np.linspace(0, T, int(fs * T), endpoint=False)
print(f"\nAsse temporale: {len(t)} campioni, da {t[0]:.3f} a {t[-1]:.3f} s")
print(f"Passo:          {t[1]-t[0]:.4f} s = {1000*(t[1]-t[0]):.2f} ms")

### Indexing e slicing: estrarre porzioni di segnale

Sintassi identica alle liste Python, ma più potente.

In [ ]:
import numpy as np

# Segnale di esempio: 10 campioni
x = np.linspace(0, 9, 10)
print(f"x = {x}")

# Accesso singolo (come C)
print(f"x[0]    = {x[0]}")    # primo elemento
print(f"x[-1]   = {x[-1]}")   # ultimo elemento
print(f"x[-3]   = {x[-3]}")   # terzultimo

# Slicing: x[start:stop:step] — stop ESCLUSO
print(f"x[:3]   = {x[:3]}")   # primi 3
print(f"x[7:]   = {x[7:]}")   # dal settimo in poi
print(f"x[2:6]  = {x[2:6]}")  # dal 2 al 5
print(f"x[::2]  = {x[::2]}")  # uno su due (downsampling)
print(f"x[::-1] = {x[::-1]}") # invertito

# Caso pratico: estrarre i primi 100 ms di un segnale
fs = 1000
t = np.linspace(0, 1.0, fs, endpoint=False)
segnale = np.sin(2 * np.pi * 50 * t)  # sinusoide 50 Hz
primi_100ms = segnale[:int(0.1 * fs)]
print(f"\nSegnale completo:   {len(segnale)} campioni (1.0 s)")
print(f"Primi 100 ms:       {len(primi_100ms)} campioni")

### Operazioni vettoriali: addio `for`

Con NumPy le operazioni aritmetiche si applicano **elemento per elemento** automaticamente.

In [ ]:
import numpy as np

fs = 1000
t  = np.linspace(0, 1.0, fs, endpoint=False)

# Somma di due sinusoidi — NESSUN ciclo for!
f1, f2 = 50, 120       # Hz
A1, A2 = 1.0, 0.5      # ampiezze

s1 = A1 * np.sin(2 * np.pi * f1 * t)
s2 = A2 * np.sin(2 * np.pi * f2 * t)
s  = s1 + s2

# Operazioni su tutto il segnale in una riga
s_gain    = 10.0 * s                          # guadagno 20 dB
s_offset  = s + 1.65                          # aggiunta di offset DC
s_clipped = np.clip(s, -0.8, 0.8)             # hard clipping
s_power   = s ** 2                             # potenza istantanea
s_rms     = np.sqrt(np.mean(s ** 2))           # valore RMS

print(f"Segnale s: min={s.min():.3f}, max={s.max():.3f}, RMS={s_rms:.3f}")
print(f"Clippato:  min={s_clipped.min():.3f}, max={s_clipped.max():.3f}")
print(f"Potenza media: {np.mean(s_power):.4f} W (su 1 Ω)")

### Broadcasting: operazioni tra array di dimensioni diverse

In [ ]:
import numpy as np

# Caso 1: scalare × vettore (già visto)
guadagno_db = 20.0
guadagno_lin = 10 ** (guadagno_db / 20)
segnale = np.array([0.1, 0.2, 0.5, 1.0])
print(f"Input:  {segnale}")
print(f"Output: {guadagno_lin * segnale}")

# Caso 2: 4 canali con guadagni diversi — matrice 1000x4
np.random.seed(42)
dati_multicanale = np.random.randn(1000, 4)   # 1000 campioni, 4 canali
guadagni = np.array([1.0, 2.0, 0.5, 10.0])    # uno per canale

# Broadcasting: (1000,4) × (4,) → si moltiplica colonna per colonna
dati_amplificati = dati_multicanale * guadagni

print(f"\nShape input:  {dati_multicanale.shape}")
print(f"Shape gains:  {guadagni.shape}")
print(f"Shape output: {dati_amplificati.shape}")

# Verifica: RMS per canale prima e dopo
rms_prima = np.sqrt(np.mean(dati_multicanale**2, axis=0))
rms_dopo  = np.sqrt(np.mean(dati_amplificati**2, axis=0))
print(f"\nRMS per canale prima: {rms_prima}")
print(f"RMS per canale dopo:  {rms_dopo}")
print(f"Rapporto (≈ guadagni): {rms_dopo / rms_prima}")

### Funzioni universali (ufunc): il toolkit matematico

In [ ]:
import numpy as np

fs  = 5000
t   = np.linspace(0, 0.01, int(fs * 0.01), endpoint=False)

# Funzioni trigonometriche
s_sin  = np.sin(2 * np.pi * 100 * t)
s_cos  = np.cos(2 * np.pi * 100 * t)

# Funzioni esponenziali e logaritmiche
tau    = 0.003                         # costante di tempo 3 ms
s_exp  = np.exp(-t / tau)              # decadimento esponenziale
s_log  = 20 * np.log10(np.abs(s_sin) + 1e-10)  # in dB (+ epsilon per log(0))

# Generare un chirp lineare (swept sine)
f_start, f_stop = 10, 500             # Hz
chirp_phase = 2 * np.pi * (f_start * t + (f_stop - f_start) / (2 * t[-1]) * t**2)
s_chirp = np.sin(chirp_phase)

# Statistiche
print(f"Segnale sin: mean={np.mean(s_sin):.6f}, std={np.std(s_sin):.4f}, RMS={np.sqrt(np.mean(s_sin**2)):.4f}")
print(f"Chirp:       da {f_start} a {f_stop} Hz in {t[-1]*1000:.1f} ms, {len(s_chirp)} campioni")

### Shape, reshape e arrays multidimensionali

In [ ]:
import numpy as np

# Segnale acquisito da 4 canali, 256 campioni cadauno (flat)
n_ch, n_samp = 4, 256
dati_flat = np.random.randn(n_ch * n_samp)
print(f"Dati flat: shape={dati_flat.shape}")

# Reshape in matrice canali × campioni
dati_2d = dati_flat.reshape(n_ch, n_samp)
print(f"Dati 2D:   shape={dati_2d.shape}  → {n_ch} canali × {n_samp} campioni")

# Operazioni per asse
# axis=0 → opera lungo i canali (per ogni campione)
# axis=1 → opera lungo i campioni (per ogni canale)
media_per_canale  = np.mean(dati_2d, axis=1)  # un valore per canale
media_per_campione= np.mean(dati_2d, axis=0)  # un valore per campione
print(f"\nMedia per canale:   {media_per_canale}")   # shape (4,)
print(f"Shape media/samp:   {media_per_campione.shape}")  # shape (256,)

# Trasposta
dati_T = dati_2d.T   # ora: campioni × canali
print(f"\nTrasposta: {dati_T.shape}  → {n_samp} campioni × {n_ch} canali")

### Maschere booleane: filtrare per condizione

In [ ]:
import numpy as np

np.random.seed(0)
fs = 1000
t  = np.linspace(0, 1.0, fs, endpoint=False)
s  = np.sin(2 * np.pi * 5 * t) + 0.1 * np.random.randn(fs)

# Maschera booleana: True dove il segnale supera la soglia
soglia = 0.7
maschera = s > soglia
print(f"Campioni sopra {soglia}: {np.sum(maschera)} su {fs}")
print(f"Percentuale:            {100 * np.mean(maschera):.1f}%")

# Estrarre i valori (fancy indexing)
campioni_alti = s[maschera]
print(f"Valori estratti:        {campioni_alti[:5]} ...")

# Combinare condizioni (& = AND, | = OR, ~ = NOT)
# Campioni nell'intervallo [200ms, 600ms] CON ampiezza > 0
finestra = (t >= 0.2) & (t <= 0.6) & (s > 0)
print(f"\nCampioni in finestra positiva: {np.sum(finestra)}")

# Sostituire i valori fuori soglia (saturazione hardware)
s_saturato = s.copy()
s_saturato[s_saturato >  0.8] =  0.8
s_saturato[s_saturato < -0.8] = -0.8
print(f"\nMax dopo saturazione: {s_saturato.max():.3f} (era {s.max():.3f})")

---
## 🔧 Esercizio intermedio: Il generatore di segnali

Scrivi una funzione `genera_segnale(tipo, freq, durata, fs, snr_db=None)` che produce i seguenti tipi di segnale, con l'aggiunta opzionale di rumore gaussiano.

In [ ]:
import numpy as np

def genera_segnale(tipo, freq, durata, fs, snr_db=None):
    """
    Genera un segnale periodico campionato.

    Parametri
    ----------
    tipo    : str — 'sin', 'square', 'sawtooth', 'chirp'
    freq    : float — frequenza fondamentale in Hz
    durata  : float — durata in secondi
    fs      : float — frequenza di campionamento in Hz
    snr_db  : float o None — aggiunge rumore gaussiano all'SNR specificato

    Ritorna
    -------
    t, x    : array di tempo e segnale
    """
    N = int(fs * durata)
    t = np.linspace(0, durata, N, endpoint=False)
    fase = 2 * np.pi * freq * t

    if tipo == 'sin':
        x = np.sin(fase)
    elif tipo == 'square':
        x = np.sign(np.sin(fase))
    elif tipo == 'sawtooth':
        x = 2 * (t * freq - np.floor(t * freq + 0.5))
    elif tipo == 'chirp':
        f_stop = freq * 10
        chirp_phase = 2 * np.pi * (freq * t + (f_stop - freq) / (2 * durata) * t**2)
        x = np.sin(chirp_phase)
    else:
        raise ValueError(f"Tipo '{tipo}' non riconosciuto. Scegli: sin, square, sawtooth, chirp")

    if snr_db is not None:
        p_segnale = np.mean(x**2)
        p_rumore  = p_segnale / (10 ** (snr_db / 10))
        rumore    = np.random.randn(N) * np.sqrt(p_rumore)
        x = x + rumore

    return t, x

# Test rapido
for tipo in ['sin', 'square', 'sawtooth', 'chirp']:
    t, x = genera_segnale(tipo, freq=100, durata=0.05, fs=10000, snr_db=20)
    rms = np.sqrt(np.mean(x**2))
    print(f"{tipo:10s}: {len(x)} campioni, RMS={rms:.4f}, range=[{x.min():.3f}, {x.max():.3f}]")

In [ ]:
# Visualizzazione (anticipazione di Matplotlib — Lezione 6)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 5))
tipi = ['sin', 'square', 'sawtooth', 'chirp']
for ax, tipo in zip(axes.flat, tipi):
    t, x = genera_segnale(tipo, freq=100, durata=0.03, fs=10000, snr_db=30)
    ax.plot(t * 1000, x, linewidth=0.8)
    ax.set_title(tipo)
    ax.set_xlabel("Tempo [ms]")
    ax.set_ylabel("Ampiezza")
    ax.grid(True, alpha=0.3)
plt.suptitle('Generatore di segnali', fontsize=13)
plt.tight_layout()
plt.show()

---
## Blocco 2.2 — FFT e analisi spettrale (50 min)

### Dalla teoria alla pratica

La FFT che avete studiato a lezione è la DFT implementata in modo efficiente. NumPy la calcola con `np.fft.fft()`. Il risultato è un vettore di **N numeri complessi**: modulo = ampiezza, angolo = fase.

Relazione fondamentale: per un segnale di N campioni a frequenza $f_s$, la **risoluzione in frequenza** è:

$$\Delta f = \frac{f_s}{N} = \frac{1}{T_{tot}}$$

Vuoi la metà della risoluzione? Devi campionare il doppio del tempo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Segnale di test: sinusoide pura ---
fs = 1000.0    # Hz
T  = 1.0       # secondi
N  = int(fs * T)
t  = np.linspace(0, T, N, endpoint=False)

f0 = 50.0      # Hz
A  = 2.0       # ampiezza
x  = A * np.sin(2 * np.pi * f0 * t)

# --- FFT ---
X   = np.fft.fft(x)           # vettore complesso, N elementi
f   = np.fft.fftfreq(N, 1/fs) # asse in frequenza (da -fs/2 a +fs/2)

print(f"Segnale:   {N} campioni, fs={fs:.0f} Hz, T={T:.1f} s")
print(f"Risoluzione spettrale: Δf = fs/N = {fs/N:.3f} Hz")
print(f"FFT output: {X.shape}, dtype={X.dtype}")

# Il picco dovrebbe essere a f0 = 50 Hz con ampiezza N/2 * A
idx_picco = np.argmax(np.abs(X[:N//2]))
print(f"\nPicco FFT a: {f[idx_picco]:.1f} Hz")
print(f"Ampiezza complessa: |X[{idx_picco}]| = {np.abs(X[idx_picco]):.1f}")
print(f"Ampiezza reale:     A = {2 * np.abs(X[idx_picco]) / N:.3f} (atteso: {A:.1f})")

### Spettro monolatero e ampiezza corretta

La FFT produce N componenti, ma per segnali reali le frequenze negative sono il coniugato di quelle positive — non aggiungono informazione. Si usa lo **spettro monolatero** (da 0 a fs/2) con ampiezza raddoppiata.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def spettro_monolatero(x, fs):
    """
    Calcola lo spettro di ampiezza monolatero (0 a fs/2).

    Ritorna
    -------
    freqs : frequenze in Hz
    amp   : ampiezze (unità fisiche, non normalizzate)
    """
    N   = len(x)
    X   = np.fft.fft(x)
    f   = np.fft.fftfreq(N, 1/fs)

    # Solo frequenze positive (0 a fs/2)
    idx   = f >= 0
    freqs = f[idx]
    amp   = np.abs(X[idx]) * 2 / N    # ×2 perché buttiamo il lato negativo
    amp[0] /= 2                        # la DC non si raddoppia
    if N % 2 == 0:
        amp[-1] /= 2                   # Nyquist non si raddoppia
    return freqs, amp

# Test con segnale noto
fs = 2000.0
t  = np.linspace(0, 1.0, int(fs), endpoint=False)
# 3 componenti: 50 Hz (A=2.0), 150 Hz (A=1.0), 300 Hz (A=0.5)
x  = 2.0 * np.sin(2*np.pi*50*t) + 1.0 * np.sin(2*np.pi*150*t) + 0.5 * np.sin(2*np.pi*300*t)

freqs, amp = spettro_monolatero(x, fs)

# Trova i 3 picchi più alti
idx_picchi = np.argsort(amp)[-3:][::-1]
print("Componenti trovate (ordinate per ampiezza):")
for i in idx_picchi:
    print(f"  f = {freqs[i]:>6.1f} Hz,  A = {amp[i]:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(freqs, amp)
ax.set_xlabel("Frequenza [Hz]")
ax.set_ylabel("Ampiezza")
ax.set_title("Spettro monolatero")
ax.set_xlim(0, 500)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Spettro in dB

In ambito EE si lavora quasi sempre in **dB** rispetto a una riferimento. Il vantaggio è la scala logaritmica: componenti piccole vicino a quelle grandi diventano visibili.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def spettro_db(x, fs, ref=None):
    """
    Spettro di ampiezza monolatero in dB.
    ref: valore di riferimento (default: massimo del segnale = 0 dBFS)
    """
    freqs, amp = spettro_monolatero(x, fs)
    if ref is None:
        ref = amp.max()
    amp_db = 20 * np.log10(amp / ref + 1e-12)  # +epsilon evita log(0)
    return freqs, amp_db

# Segnale: 3 componenti + rumore
np.random.seed(42)
fs = 4000.0
t  = np.linspace(0, 1.0, int(fs), endpoint=False)
x  = (2.0 * np.sin(2*np.pi*100*t)
    + 0.5 * np.sin(2*np.pi*300*t)
    + 0.05 * np.sin(2*np.pi*700*t)
    + 0.02 * np.random.randn(len(t)))

freqs, amp_db = spettro_db(x, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

# Scala lineare — la componente a 700 Hz è quasi invisibile
freqs_lin, amp_lin = spettro_monolatero(x, fs)
ax1.plot(freqs_lin, amp_lin)
ax1.set_title("Scala lineare — componenti piccole invisibili")
ax1.set_xlabel("Frequenza [Hz]")
ax1.set_ylabel("Ampiezza")
ax1.grid(True, alpha=0.3)

# Scala dB — ora si vede tutto
ax2.plot(freqs, amp_db)
ax2.set_title("Scala logaritmica (dBFS) — componenti piccole visibili")
ax2.set_xlabel("Frequenza [Hz]")
ax2.set_ylabel("Ampiezza [dBFS]")
ax2.set_ylim(-80, 5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Finestratura: il leakage spettrale

La FFT assume che il segnale sia **periodico** nel blocco analizzato. Se la frequenza del segnale non è un multiplo esatto di Δf, il segnale viene troncato bruscamente → **leakage**: l'energia si spande sulle frequenze vicine.

La soluzione è **moltiplicare il segnale per una finestra** che porta a zero i bordi del blocco prima di calcolare la FFT.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 1000.0
N  = 1000
t  = np.linspace(0, 1.0, N, endpoint=False)

# Frequenza NON multipla di Δf=1 Hz → leakage
f0 = 50.7
x  = np.sin(2 * np.pi * f0 * t)

# Finestre disponibili in NumPy
finestre = {
    'Rettangolare':       np.ones(N),
    'Hanning':            np.hanning(N),
    'Hamming':            np.hamming(N),
    'Blackman':           np.blackman(N),
    'Blackman-Harris':    np.blackman(N),  # approssimazione
}

fig, axes = plt.subplots(1, len(finestre), figsize=(14, 4), sharey=True)

for ax, (nome, win) in zip(axes, finestre.items()):
    x_win = x * win
    X     = np.fft.fft(x_win)
    f     = np.fft.fftfreq(N, 1/fs)
    idx   = f >= 0
    amp_db = 20 * np.log10(np.abs(X[idx]) / (N/2) + 1e-12)
    ax.plot(f[idx], amp_db, linewidth=0.8)
    ax.set_title(nome, fontsize=9)
    ax.set_xlim(30, 80)
    ax.set_ylim(-80, 5)
    ax.set_xlabel("Hz")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("dBFS")
plt.suptitle(f'Effetto della finestra su f₀={f0} Hz (leakage)', fontsize=12)
plt.tight_layout()
plt.show()

print("Confronto:")
print("  Rettangolare:  lobi laterali alti (-13 dB), lobo principale stretto")
print("  Hanning:       buon compromesso (-32 dB sidelobes)")
print("  Blackman:      sidelobes molto bassi (-58 dB), lobo più largo")

### Zero-padding: più risoluzione (apparente)

Aggiungere zeri al segnale prima della FFT **non aumenta la risoluzione fisica** (quella dipende solo dalla durata del segnale), ma **interpola lo spettro** producendo una curva più fluida — utile per individuare con precisione la posizione di un picco.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 1000.0
N  = 64          # blocco corto → scarsa risoluzione
t  = np.linspace(0, N/fs, N, endpoint=False)
f0 = 100.0
x  = np.sin(2 * np.pi * f0 * t)

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
fattori = [1, 4, 16, 64]

for ax, pad in zip(axes, fattori):
    x_pad = np.zeros(N * pad)
    x_pad[:N] = x
    X   = np.fft.fft(x_pad)
    f   = np.fft.fftfreq(N * pad, 1/fs)
    idx = (f >= 0) & (f <= 300)
    amp = 2 * np.abs(X[idx]) / N  # normalizza rispetto al N originale
    ax.plot(f[idx], amp, marker='.', markersize=3, linewidth=0.8)
    ax.axvline(f0, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_title(f'×{pad} (N={N*pad})')
    ax.set_xlabel("Hz")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Ampiezza")
plt.suptitle('Zero-padding: stessa risoluzione fisica, più punti di interpolazione', fontsize=11)
plt.tight_layout()
plt.show()

---
## 🔧 Esercizio di lezione: Analizzatore di spettro

Il segnale qui sotto è la somma di **3 sinusoidi** a frequenze ignote più **rumore gaussiano**. Il tuo compito è identificare le componenti e caratterizzare il segnale.

> Usa le funzioni `genera_segnale`, `spettro_monolatero` e `spettro_db` definite in precedenza — riesegui quelle celle prima di procedere.

In [ ]:
import numpy as np

# Segnale misterioso — NON modificare questa cella
np.random.seed(7)
fs   = 8000.0
t    = np.linspace(0, 1.0, int(fs), endpoint=False)
x_segreto = (  1.50 * np.sin(2*np.pi * 120  * t)
             + 0.80 * np.sin(2*np.pi * 440  * t + np.pi/4)
             + 0.30 * np.sin(2*np.pi * 1800 * t)
             + 0.05 * np.random.randn(len(t)))
print(f"Segnale: {len(x_segreto)} campioni, fs={fs:.0f} Hz, durata=1.0 s")
print(f"Range:   [{x_segreto.min():.3f}, {x_segreto.max():.3f}]")

### Parte 1 — Spettro di ampiezza in dB
Calcola e plotta lo spettro monolatero in dBFS del segnale misterioso.

In [ ]:
# ============================================
# PARTE 1 — Spettro in dB
# ============================================

# HINT:
# freqs, amp_db = spettro_db(x_segreto, fs)
# poi plt.plot(freqs, amp_db)
# Limita l'asse x a [0, 3000] Hz per vedere le componenti

### Parte 2 — Identificare le 3 frequenze
Trova i 3 picchi dominanti e le loro ampiezze lineari.

In [ ]:
# ============================================
# PARTE 2 — Trovare i picchi
# ============================================

# HINT: usa np.argsort() sull'array amp lineare
# per trovare gli indici dei 3 valori più alti
# freqs, amp = spettro_monolatero(x_segreto, fs)
# idx_top3 = np.argsort(amp)[-3:][::-1]
# for i in idx_top3: print(freqs[i], amp[i])

### Parte 3 — Stimare l'SNR
Lo SNR è il rapporto tra la potenza delle 3 componenti e la potenza del rumore.
Approccio semplificato: identifica una banda di frequenze dove c'è solo rumore (es. 3000–4000 Hz) e una dove ci sono i segnali.

In [ ]:
# ============================================
# PARTE 3 — Stima SNR
# ============================================

# HINT:
# freqs, amp = spettro_monolatero(x_segreto, fs)
#
# Potenza segnale: somma di amp[intorno ai picchi]^2
# Potenza rumore:  media di amp[banda_rumore]^2
#
# SNR_db = 10 * np.log10(P_segnale / P_rumore)
# Confronta con il valore 'vero': 0.05^2 rumore, segnali 1.5, 0.8, 0.3

### Parte 4 — Confronto finestre
Ripeti l'analisi con finestra rettangolare e di Hanning.
Quale finestra rende più facile distinguere la componente a 1800 Hz dal rumore?

In [ ]:
# ============================================
# PARTE 4 — Confronto finestre
# ============================================

# HINT:
# win_rect = np.ones(len(x_segreto))
# win_hann = np.hanning(len(x_segreto))
# Applica: x_win = x_segreto * win
# Poi chiama spettro_db su x_win
# Attenzione: con finestra la normalizzazione cambia leggermente
# (dividi per np.sum(win)/2 invece di N/2)

---
## ✅ Soluzioni

Esegui la cella qui sotto solo dopo aver provato.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Rigenera il segnale (nel caso questa cella venga eseguita da sola)
np.random.seed(7)
fs = 8000.0
t  = np.linspace(0, 1.0, int(fs), endpoint=False)
x_segreto = (  1.50 * np.sin(2*np.pi * 120  * t)
             + 0.80 * np.sin(2*np.pi * 440  * t + np.pi/4)
             + 0.30 * np.sin(2*np.pi * 1800 * t)
             + 0.05 * np.random.randn(len(t)))

# Riesegui le funzioni definite sopra se necessario
def spettro_monolatero(x, fs):
    N = len(x); X = np.fft.fft(x); f = np.fft.fftfreq(N, 1/fs)
    idx = f >= 0; freqs = f[idx]
    amp = np.abs(X[idx]) * 2 / N; amp[0] /= 2
    if N % 2 == 0: amp[-1] /= 2
    return freqs, amp

def spettro_db(x, fs, ref=None):
    freqs, amp = spettro_monolatero(x, fs)
    if ref is None: ref = amp.max()
    return freqs, 20 * np.log10(amp / ref + 1e-12)

# ── Parte 1 ──────────────────────────────────────────────────
freqs, amp_db = spettro_db(x_segreto, fs)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(freqs, amp_db, linewidth=0.8)
ax.set_xlabel("Frequenza [Hz]"); ax.set_ylabel("dBFS")
ax.set_title("Spettro di ampiezza — segnale misterioso")
ax.set_xlim(0, 3000); ax.set_ylim(-60, 5); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# ── Parte 2: trovare i picchi ────────────────────────────────
freqs, amp = spettro_monolatero(x_segreto, fs)
idx_top3 = np.argsort(amp)[-3:][::-1]
print("=== Componenti trovate ===")
for i in idx_top3:
    print(f"  f = {freqs[i]:>6.0f} Hz,  A = {amp[i]:.4f}")
print("\nValori veri: 120 Hz (A=1.5), 440 Hz (A=0.8), 1800 Hz (A=0.3)")

# ── Parte 3: SNR ────────────────────────────────────────────
# Banda segnale: intorno ai 3 picchi (±10 Hz)
P_segnale = 0.0
for i in idx_top3:
    maschera_pic = np.abs(freqs - freqs[i]) < 10
    P_segnale += np.sum(amp[maschera_pic]**2)

# Banda di solo rumore: 3000-4000 Hz
banda_rumore = (freqs >= 3000) & (freqs <= 4000)
P_rumore_spettrale = np.mean(amp[banda_rumore]**2)
n_bin_totali = int(fs / 2 / (fs / len(x_segreto)))  # numero di bin 0-fs/2
P_rumore = P_rumore_spettrale * n_bin_totali
SNR_db = 10 * np.log10(P_segnale / P_rumore)
print(f"\n=== SNR stimato === {SNR_db:.1f} dB")
# SNR teorico: P_segnale = (1.5^2 + 0.8^2 + 0.3^2)/2, P_rumore = 0.05^2
P_s_teo = (1.5**2 + 0.8**2 + 0.3**2) / 2
P_r_teo = 0.05**2
print(f"SNR teorico:  {10*np.log10(P_s_teo/P_r_teo):.1f} dB")

# ── Parte 4: confronto finestre ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (nome, win) in zip(axes, [('Rettangolare', np.ones(len(x_segreto))),
                                   ('Hanning',      np.hanning(len(x_segreto)))]):
    x_win = x_segreto * win
    N = len(x_win)
    X = np.fft.fft(x_win)
    f = np.fft.fftfreq(N, 1/fs)
    idx = f >= 0
    amp_db_w = 20 * np.log10(np.abs(X[idx]) / (np.sum(win)/2) + 1e-12)
    ax.plot(f[idx], amp_db_w, linewidth=0.8)
    ax.set_title(nome)
    ax.set_xlim(0, 3000); ax.set_ylim(-60, 5)
    ax.set_xlabel("Hz"); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("dB")
plt.suptitle('Confronto finestre — dettaglio 0-3000 Hz', fontsize=12)
plt.tight_layout(); plt.show()

---
## Riepilogo della Lezione 2

| Concetto | Funzione chiave | Nota |
|----------|----------------|------|
| Creare array | `np.linspace`, `np.arange`, `np.zeros` | `endpoint=False` per asse tempo |
| Slicing | `x[start:stop:step]` | `stop` escluso, indici negativi |
| Operazioni vettoriali | `+`, `*`, `**`, `np.sin()` | Niente `for`! |
| Broadcasting | `arr * scalare`, `matrice * vettore` | Dimensioni compatibili |
| Maschere booleane | `arr[arr > soglia]` | Combinare con `&`, `\|`, `~` |
| FFT | `np.fft.fft(x)` | Output complesso, N punti |
| Asse in frequenza | `np.fft.fftfreq(N, 1/fs)` | Da -fs/2 a +fs/2 |
| Spettro monolatero | `amp = 2*abs(X[:N//2])/N` | Ampiezza fisica corretta |
| Scala dB | `20*np.log10(amp/ref)` | +epsilon per evitare log(0) |
| Finestratura | `np.hanning(N)`, `np.blackman(N)` | Moltiplica prima della FFT |
| Zero-padding | `np.zeros(N*pad); x_pad[:N]=x` | Interpola, non risolve |

### Nella prossima lezione
Passeremo a **SciPy**: progettare filtri digitali, stimare la PSD con il metodo di Welch, trovare picchi automaticamente e fittare curve sui dati sperimentali.